In [ ]:
using GridapTopOpt, Gridap, Gridap.TensorValues 

In [ ]:
const E = 30e3        # Young's modulus
const ν = 0.3         # Poisson's ratio
const G = E/(2*(1+ν)) # Shear modulus
const α_mat = 12e-6   #(Thermal Expansion)
const κ_mat =  1.0    #(Thermal conductivity)

const Height = 10                 # Height of the specimen
const Heigthfact = 1.5            # Height factor for length
const Length = Heigthfact*Height  # Length of the half specimen
const Widthfact = 1/8             # Width factor for Width
const Width = Height*Widthfact    # Width of the specimen
const Loadfact = Length/40        # Load path on the half specimen
const HeatSource_L = Length/30    # Heat source left 
const HeatSource_H = Height/30    # Heat source right

const l = Height/5 # Modified     # Bending length scale
const N = 0.5                     # Micropolar coupling number

const lt = l                      # Torsional length scale
const χ = 1.0                     # Polar ratio

const λₘₐₜ = 2*G*ν/(1 -2*ν)          # Different micropolar material properties
const κₘₐₜ = 2*G*N^2/(1-N^2)
const μₘₐₜ = G*(1-2*(N^2))/(1-N^2)
const γₘₐₜ = 4*G*l^2
const βₘₐₜ = 2*G*(lt^2 -2*l^2)
const αₘₐₜ = 2*G*lt^2*((1/χ) -1)

const  T0 = 0          # Initial temperature in the domain
const  TApp = 20.0     # Applied temperature at the boundary
const  Qh = 1.0        # Not used

const fApp = 4         # Mechanical load
g(x) = VectorValue(0.0,-1.0,0.0) * (1 - x[3]/Width) * fApp   # Applied out-of-plane bending and torsion

In [ ]:
# FE parameters
order = 1                                                              # Finite element order
dom = (0,Length,0,Height,0,Width)                                      # Bounding domain
nx,ny,nz = (60,40,5) 
el_size = (nx,ny,nz)#(300,100)                                         # Mesh partition size

In [ ]:
model = CartesianDiscreteModel(dom, el_size) # Initialize the model
el_Δ = get_el_Δ(model)
f_Γ_D_1(x) = (x[1] ≈ 0.0) # Left side
f_Γ_D_2(x) = (x[1] ≈ Length)      # Right side                                                                                 
f_Γ_D_3(x) = (x[2] ≈ 0.0) && ((x[1] < Length - Length/20)) #|| (x[1] > Length/2 + Length/40)) # Bottom face
f_Γ_D_4(x) = (x[2] ≈ Height)  # Top face
f_Γ_D_5(x) = (Length/2 - HeatSource_L <= x[1] <= Length/2 + HeatSource_L) && (Height/2 - HeatSource_H <= x[2] <= Height/2 + HeatSource_H) #Left heat source
                                                                                
f_Γ_N(x) = (x[2] ≈ 0.0) && (Length - Length/20 <= x[1] )# <= Length/2 + Length/40 )            # Γ_N indicator function

update_labels!(1,model,f_Γ_D_1,"Gamma_D_1")                                 

update_labels!(3,model,f_Γ_D_2,"Gamma_D_2")

update_labels!(4,model,f_Γ_D_3,"Gamma_D_3")

update_labels!(5,model,f_Γ_D_4,"Gamma_D_4")

update_labels!(6,model,f_Γ_D_5,"Gamma_D_5")

update_labels!(2,model,f_Γ_N,"Gamma_N")

In [ ]:
# FD parameters
γ = 0.1                                                            # HJ eqn time step coeff
γ_reinit = 0.5                                                 # Reinit. eqn time step coeff
max_steps = 14#floor(Int,order*maximum(el_size)/5)                 # Max steps for advection   
tol = 0.001#1/(5order^2)/minimum(el_size)                          # Reinitialisation tolerance                                   

In [ ]:
I2 = SymTensorValue{3,Float64}(1.0, 0.0, 0.0, 1.0, 0.0, 1.0)

function σ_Tot(ε)
    σM = (λₘₐₜ*tr(ε)*one(ε) + (2*μₘₐₜ + κₘₐₜ)*(ε))
    return σM
end

E_Matrix = zeros(3,3,3)
E_Matrix[1,2,3]=1.0
E_Matrix[2,3,1]=1.0
E_Matrix[3,1,2]=1.0
E_Matrix[1,3,2]=-1.0
E_Matrix[3,2,1]=-1.0
E_Matrix[2,1,3]=-1.0
E_Tensor = ThirdOrderTensorValue(E_Matrix ...)

function σ_Temp(T)
    ε_th = -α_mat * T * I2
    σM = (λₘₐₜ*tr(ε_th)*one(ε_th) + (2*μₘₐₜ + κₘₐₜ)*(ε_th))
    return σM
end

function σ_Temp0(T0)
    ε_th = -α_mat * T0 * I2
    σM = (λₘₐₜ*tr(ε_th)*one(ε_th) + (2*μₘₐₜ + κₘₐₜ)*(ε_th))
    return σM
end

function k_gradTemp(∇)
    k_grad = κ_mat * (∇)
    return k_grad
end


function ε_Skw(∇,θ)
    ∇ᵀ = transpose(∇)
    w = (0.5*(∇ᵀ - ∇)) - (E_Tensor⋅θ)
    return w
end

function σ_Cmod(ϵ_skew)
    σM = κₘₐₜ*ϵ_skew
    return σM
end

function M_mod(∇)
    M = αₘₐₜ*(∇ ⊙ I2)*I2 + βₘₐₜ*∇ + γₘₐₜ*∇
    return M
end

function Skw(u,θ)
    ∇ᵀ = transpose(∇(u))
    w = (0.5*(∇ᵀ - ∇(u)) - (E_Tensor⋅θ))
    return w
end


In [ ]:
# Problem parameters                                       
vf = 0.5                                                         # Volume fraction constraint
lsf_func = initial_lsf((12/Length),0.2)                          # Initial level set function  (12/Length)
iter_mod = 10                                                    # VTK Output modulo                        
path = "./FixedBeamNew3/Result_tol&Max_stepMod/TApp-$TApp,fApp-$fApp,Qh-$Qh/Mesh-$nx,$ny,$nz/N_$N;χ_$χ;l_$l;lt_$lt/Dim_$Length;$Height;$Width/E_$E/"      # Output path
mkpath(path)                                                     # Create path

In [ ]:
writevtk(model,path*"FixedBeam")

## Triangulations and measures
Ω = Triangulation(model)
dΩ = Measure(Ω,2*order)
Γ_N = BoundaryTriangulation(model,tags="Gamma_N")
dΓ_N = Measure(Γ_N,2*order)
Γ_N_Therm = BoundaryTriangulation(model,tags=["Gamma_D_5"])
dΓ_N_Therm = Measure(Γ_N_Therm,2*order)
vol_D = sum(∫(1)dΩ)

In [ ]:
## Spaces
reffe = ReferenceFE(lagrangian,VectorValue{3,Float64},order)
reffe_scalar = ReferenceFE(lagrangian,Float64,order)
V = TestFESpace(model,reffe;conformity=:H1,dirichlet_tags=["Gamma_D_1","Gamma_D_2"],dirichlet_masks = [(true, true,true),(true, false ,true)])
U = TrialFESpace(V,[VectorValue(0.0,0.0,0.0),VectorValue(0.0,0.0,0.0)])   ## Dispalcement Space

R = TestFESpace(model,reffe;conformity=:H1,dirichlet_tags=["Gamma_D_1","Gamma_D_2"],dirichlet_masks = [(true, true, true),(true, false, true)])
S = TrialFESpace(R,[VectorValue(0.0,0.0,0.0),VectorValue(0.0,0.0,0.0)]) # theta(Micro-Rotation)

Q = TestFESpace(model,reffe_scalar;conformity=:H1,dirichlet_tags=["Gamma_D_1","Gamma_D_3","Gamma_N","Gamma_D_4","Gamma_D_5"],dirichlet_masks = [(true),(true),(true),(true),(true)])
P = TrialFESpace(Q,[T0,TApp,TApp,T0,TApp]) 

In [ ]:
USP = MultiFieldFESpace([U,S,P])
VRQ = MultiFieldFESpace([V,R,Q])

In [ ]:
V_φ = TestFESpace(model,reffe_scalar)
V_reg = TestFESpace(model,reffe_scalar)
U_reg = TrialFESpace(V_reg)

In [ ]:
# Level set and interpolator
φh = interpolate(lsf_func,V_φ)
interp = SmoothErsatzMaterialInterpolation(η = (2)*maximum(get_el_Δ(model)), ϵ = 1e-9)    # η = 2 ×  maximum side length of an element.
interp2 = SmoothErsatzMaterialInterpolation(η = (2)*maximum(get_el_Δ(model)), ϵ = 0.03)    # η = 2 ×  maximum side length of an element.
I,H,DH,ρ = interp.I,interp.H,interp.DH,interp.ρ
I1 = interp2.I

In [ ]:
writevtk(Ω,path*"initial_lsf",cellfields=["phi"=>φh,
  "ρ(phi)"=>(ρ ∘ φh),"|nabla(phi)|"=>(norm ∘ ∇(φh))])

In [ ]:
a((u,θ,T),(w,v,z),φ)  = ∫((I ∘ φ)*( (ε(w) ⊙ (σ_Tot∘(ε(u)))) +  (ε(w) ⊙ (σ_Temp∘(T)))  
                                             + ((∇(v))⊙ (M_mod∘(∇(θ)))) + ((Skw(w,v)) ⊙ (σ_Cmod∘(ε_Skw∘(∇(u),θ))) )
                                            - ((v ⋅ ((E_Tensor) ⋅² (σ_Cmod∘(ε_Skw∘(∇(u),θ))))) )))dΩ  + ∫((I1 ∘ φ)*(∇(z) ⋅ (k_gradTemp∘(∇(T)))))dΩ 

lm((w,v,z),φ)= ∫((I ∘ φ)*(ε(w) ⊙ (σ_Temp0(T0))))dΩ + ∫(w ⋅ g)dΓ_N #+ ∫(z * Qh)dΓ_N_Therm

In [ ]:
state_map = RepeatingAffineFEStateMap(1, a, [lm],USP,VRQ,V_φ)#,U_reg,φh,dΩ,dΓ_N,dΓ_N_Therm)

In [ ]:
evo = FiniteDifferenceEvolver(FirstOrderStencil(3,Float64),model,V_φ;max_steps)
reinit = FiniteDifferenceReinitialiser(FirstOrderStencil(3,Float64),model,V_φ;tol,γ_reinit)
ls_evo = LevelSetEvolution(evo,reinit)

In [ ]:
C2 = isotropic_elast_tensor(3,30e3,0.3)

In [ ]:
function Cᴴ(r,s,uϕ,φ,dΩ,dΓ_N,dΓ_N_Therm)
    u_s = uϕ[2s-1]; θ_s = uϕ[2s]; T_s = uϕ[2s+1]
    ∫((I ∘ φ)*(C2 ⊙ (ε(u_s) - α_mat*(T_s - T0)*I2) ⊙ (ε(u_s) - α_mat*(T_s - T0)*I2)))dΩ #∫((u_s)·g)dΓ_N 
end


J(uϕ,φ) = 1*Cᴴ(1,1,uϕ,φ,dΩ,dΓ_N,dΓ_N_Therm)
C1(uϕ,φ) = ∫(((ρ ∘ φ) - vf)/vol_D)dΩ;

In [ ]:
DC1(q,uϕ,φ) = ∫(-1/vol_D*q*(DH ∘ φ)*(norm ∘ ∇(φ)))dΩ

In [ ]:
pcfs = PDEConstrainedFunctionals(J,[C1],state_map,analytic_dJ=nothing,analytic_dC=[DC1])

In [ ]:
α = 4max_steps*γ*maximum(get_el_Δ(model))
a_hilb(p,q) = ∫(α^2*∇(p)⋅∇(q) + p*q)dΩ;
vel_ext = VelocityExtension(a_hilb,U_reg,V_reg)

In [ ]:
function mine_al_converged(
  m::AugmentedLagrangian;
  L_tol = 1e-6,
  C_tol = 1e-3,
  window = 5
)
  h = m.history
  it = h.niter
  if it <= window
    return false
  end

  # objective stability
  Li = h[:L,it]
  L_prev = h[:L,it-window:it-1]
  A = abs(Li - mean(L_prev)) / max(abs(Li), 1.0) < L_tol

  # constraint satisfaction
  Ci = h[:C,it]
  B = maximum(abs.(Ci)) < C_tol

  return A && B
end

In [ ]:
optimiser = AugmentedLagrangian(pcfs,ls_evo,vel_ext,φh;maxiter = 450,γ,verbose=true,constraint_names=[:Vol],converged=mine_al_converged)  

In [ ]:
for (it,uh,φh) in optimiser
    uv,θv,Tv = uh
    data = ["φ"=>φh,"H(φ)"=>(H ∘ φh),"|∇(φ)|"=>(norm ∘ ∇(φh)),"uv"=>uv,"θv"=>θv,"Tv"=>Tv,"ρ(φ)"=>(ρ ∘ φh)]
    iszero(it % iter_mod) && writevtk(Ω,path*"out$it",cellfields= data) 
    write_history(path*"/historymodified$tol,max_steps-$max_steps.txt",optimiser.history)
end

In [ ]:
it = get_history(optimiser).niter; uh = get_state(pcfs);
uv,θv,Tv = uh
writevtk(Ω,path*"out$it;max_step-$max_steps;tol-$tol.vtu",cellfields=["φ"=>φh,"H(φ)"=>(H ∘ φh),"|∇(φ)|"=>(norm ∘ ∇(φh)),"uv"=>uv,"θv"=>θv,"Tv"=>Tv,"ρ(φ)"=>(ρ ∘ φh)])